# Day 6 — Hands-On Lab: Star Schema Design

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 1 (Business Process Mapping) &middot; ILT 2 (Grain & Dimension Design) &middot; ILT 3 (Fact Tables Deep Dive) |
| **Duration** | 60 minutes |
| **Reads from** | `gbmart.silver.*` (built Day 5) |
| **Writes to** | `gbmart.gold.*` (Unity Catalog managed Delta tables) |
| **Builds today** | `dim_customer`, `dim_product`, `dim_date`, `dim_address`, `dim_payment_method`, `dim_orders` |
| **Does NOT build today** | `fact_sales` &mdash; that is Day 7's **Build Gold Layer** hands-on, the very next thing on the calendar |

### Learning Objectives
- Write the `fact_sales` grain sentence and confirm your dimension plan against it
- Design each of the 6 dimensions: source table, natural key, surrogate key
- Understand why `dim_customer`/`dim_product` **reuse** the surrogate key Silver already generated, while `dim_address`/`dim_payment_method`/`dim_orders` generate a fresh one here in Gold
- Generate a `dim_date` date spine — sized to the data, not an arbitrary range — with calendar attributes
- Build and verify all 6 Gold dimension tables in Unity Catalog

---
**Instructions:** Run each cell with **Shift + Enter**. Fill in any `_______` blanks as you go.

> **Schema note:** This notebook assumes `gbmart.silver.customers`, `.products`, `.orders`, `.order_items`, `.payments`, `.payment_methods`, and `.address` exist from Day 5, with snake_case columns (`customer_id`, `product_id`, ...) matching this course's cleaned-column convention. **Every build section below inspects the real schema with `printSchema()`/`display()` before selecting columns** &mdash; if your actual Day 5 output uses different names, that inspection cell shows you the truth. Adjust the `select()` calls immediately below it to match, and the rest of the notebook works unchanged.

---
## Setup: Unity Catalog Schemas

No storage account key needed here — Unity Catalog manages access to `silver` and `gold` transparently. We only need catalog/schema names.

In [ ]:
# ─── Unity Catalog Setup ─────────────────────────────────────────────────────
CATALOG = "gbmart"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

print(f"Silver schema : {SILVER}")
print(f"Gold schema   : {GOLD}  (created if it didn't exist)")

---
## Phase A — Design Before You Build

**Goal:** Put the design from this morning's ILT sessions on paper (well, in markdown) before writing a single line of PySpark.

### A1 — Write the Grain Sentence

From ILT2: *"declare the grain in one sentence before writing any code."*

**Q: What is the grain of `fact_sales`?**

*Your answer:* _______________

> Hint: it is not "one row per order." Look at what `order_items.csv` actually contains — `OrderItemID, OrderID, ProductID, Quantity`. One order can have many of these rows.

### A2 — The 6 Dimensions — Fill In Before You Code

For each dimension, fill in the source Silver table and the natural key based on this morning's ILT2. You'll confirm every answer against the real code in Phases B and C.

| Dimension | Source (Silver table) | Natural Key | Surrogate Key |
|---|---|---|---|
| `dim_customer` | _______________ | _______________ | _______________ (hint: does Gold generate this one, or reuse it?) |
| `dim_product` | _______________ | _______________ | _______________ (same hint as above) |
| `dim_date` | *(no source — generated)* | _______________ | _______________ |
| `dim_address` | _______________ | _______________ | _______________ |
| `dim_payment_method` | _______________ | _______________ | _______________ |
| `dim_orders` | _______________ | _______________ | _______________ |

> Careful with `dim_payment_method` — there are two tables that sound right. Only one is a dimension.
> Careful with `dim_customer`/`dim_product` — check whether Silver already produced a surrogate key before assuming Gold has to generate one.

### A3 — Sketch the Star

```
                     dim_customer      dim_orders
                            \               /
    dim_payment_method ── fact_sales ── dim_product
                            /               \
                    dim_date          dim_address
```

`fact_sales` (built Day 7) sits in the middle: one row per order line item, holding `Quantity_purchased`, `Actual_price`, `Discounted_price`, `Sales_amount`, plus a key to each of the 6 points of this star. **Today we build the 6 points. The center gets built next.**

---
## <span style="color:#1e40af">Phase B</span> — `dim_date` (generated — no source table)

**Goal:** Build the one dimension that has no Silver source. Every source system in this course ships transactional and reference data — none of them ships you a "list of all calendar dates." You generate `dim_date` once, as a spine wide enough to cover every date you'll ever report on, including future dates for scheduling and forecasting.

In [ ]:
from pyspark.sql.functions import (
    col, min as spark_min, max as spark_max, to_date, year, quarter, month,
    date_format, dayofweek, when
)

# ─── Generate the date spine, sized to the data — not an arbitrary range ─────
# Pull the real bounds from silver.orders: earliest order placed, latest
# delivery completed. That guarantees the spine covers every date fact_sales
# will ever need to look up, with no wasted rows either side.
date_bounds = spark.table(f"{SILVER}.orders").select(
    spark_min("order_date").alias("min_date"),
    spark_max("actual_delivery_date").alias("max_date")
).collect()[0]

min_date, max_date = date_bounds["min_date"], date_bounds["max_date"]
print(f"Date range needed: {min_date} to {max_date}")

date_spine_df = spark.sql(f'''
    SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS full_date
''')

print(f"Total dates generated: {date_spine_df.count():,}")
date_spine_df.show(5)

### B1 — Derive Calendar Attributes + `date_key`

`date_key` is the **one deliberate exception** to the "reuse or hash the natural key" convention used everywhere else in this notebook. Kimball's long-standing convention for date dimensions is an integer `YYYYMMDD` key — it's human-readable in ad-hoc queries, and because it's numeric it sorts correctly in plain ascending/descending order, unlike a hash.

> **Naming matters here:** the column is `date_key`, and the plain calendar column is named `date` (not `full_date`/`date_sk`) — that's the exact name Day 7's `fact_sales` build looks up when it joins `order_date` against this table.

In [ ]:
# ─── Derive calendar attributes + date_key ───────────────────────────────────
dim_date_df = (
    date_spine_df
    .withColumn("date_key",     date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("date",         col("full_date"))
    .withColumn("day_of_week",  date_format(col("full_date"), "EEEE"))   # "Monday", "Tuesday", ...
    .withColumn("month",        month(col("full_date")))
    .withColumn("quarter",      quarter(col("full_date")))
    .withColumn("year",         year(col("full_date")))
    .withColumn("is_weekend",   dayofweek(col("full_date")).isin([1, 7]))
    .select("date_key", "date", "day_of_week", "month", "quarter", "year", "is_weekend")
)

dim_date_df.show(5, truncate=False)
dim_date_df.printSchema()

### B2 — Write `dim_date` to Gold

In [ ]:
# ─── Write dim_date to Gold ───────────────────────────────────────────────────
dim_date_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_date")

written = spark.table(f"{GOLD}.dim_date")
print(f"{GOLD}.dim_date rows : {written.count():,}")
print(f"Distinct date_key   : {written.select('date_key').distinct().count():,}")

**Q:**
1. How many total dates did you generate? _______________
2. Does distinct `date_key` count match the row count? Why must it? _______________

---
## <span style="color:#92400e">Phase C</span> — The Other 5 Dimensions

**Goal:** For each dimension: inspect the real Silver schema first, then build it. Two of these five (`dim_customer`, `dim_product`) **reuse** a surrogate key and SCD2 tracking columns that Silver already generated on Day 5 — Gold's job for those two is to *republish* Silver's already-correct, already-versioned rows, not regenerate anything. The other three (`dim_address`, `dim_payment_method`, `dim_orders`) get a fresh `sha2()`-based surrogate key generated here in Gold, since Silver never assigned them one. Order: smallest/simplest first to build confidence, then the bigger ones.

### C1 — `dim_payment_method`

**Source:** `gbmart.silver.payment_methods` — the small lookup table (~5 rows: Credit Card, UPI, Debit Card, Net Banking, Cash-on-Delivery).
**Natural key:** `payment_method_id`
**Surrogate key:** generated fresh here in Gold — Silver never assigned this table one.

> ⚠️ **Common mistake:** this dimension sources from `payment_methods` (the lookup table), **not** `payments` (the transactional table recording each order's payment event, gift-card usage, and coupon amount). `payments` is transactional detail that will eventually sit near the fact — it is not a slowly-changing dimension.

In [ ]:
# ─── Inspect Silver source FIRST ──────────────────────────────────────────────
# If your actual column names differ (e.g. PascalCase PaymentMethodID / MethodName
# instead of snake_case payment_method_id / method_name), this cell tells you —
# adjust the select() in the next cell to match. Don't guess; look.

silver_payment_methods = spark.table(f"{SILVER}.payment_methods")
silver_payment_methods.printSchema()
silver_payment_methods.display()

In [ ]:
from pyspark.sql.functions import col, sha2, concat_ws

# ─── Build dim_payment_method ─────────────────────────────────────────────────
dim_payment_method_df = (
    silver_payment_methods
    .select("payment_method_id", "method_name")
    .withColumn("payment_method_sk", sha2(col("payment_method_id").cast("string"), 256))
    .select("payment_method_sk", "payment_method_id", "method_name")
)

dim_payment_method_df.show(truncate=False)

In [ ]:
# ─── Write + verify dim_payment_method ────────────────────────────────────────
dim_payment_method_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_payment_method")

written = spark.table(f"{GOLD}.dim_payment_method")
print(f"{GOLD}.dim_payment_method rows       : {written.count()}")
print(f"Distinct payment_method_sk         : {written.select('payment_method_sk').distinct().count()}")

### C2 — `dim_customer` (SCD2 — already versioned by Silver)

**Source:** `gbmart.silver.customers`
**Natural key:** `customer_id`
**Surrogate key:** `customer_sk` — **reused from Silver**, not regenerated here.

Day 5's HOL already turned `silver.customers` into a proper SCD2 table: it generated `customer_sk`, and it stamped every row with `is_current` / `effective_start_date` / `effective_end_date`. By the time Gold reads it, the versioning decision is already made — a `MERGE`-based Gold rebuild would just be matching `customer_sk` to itself with no actual decision left to take. So Gold's job here is simple: **select the columns Gold needs and republish them with a full overwrite.** The overwrite is safe precisely because Silver, not Gold, owns the SCD2 logic.

In [ ]:
# ─── Inspect Silver source FIRST ──────────────────────────────────────────────
silver_customers = spark.table(f"{SILVER}.customers")
silver_customers.printSchema()
silver_customers.display()

In [ ]:
# ─── Build dim_customer ───────────────────────────────────────────────────────
# NOTE: customer_sk / is_current / effective_start_date / effective_end_date all
# already exist on silver.customers (Day 5 built them as part of its SCD2 load).
# We SELECT them here -- we do not call sha2() again. Regenerating the key from
# customer_id alone would silently collapse every historical version back down
# to one row per customer, throwing away the SCD2 history Silver just built.
dim_customer_df = (
    silver_customers
    .select(
        "customer_sk", "customer_id", "full_name", "email", "phone_number",
        "is_current", "effective_start_date", "effective_end_date"
    )
)

dim_customer_df.show(5, truncate=False)

In [ ]:
# ─── Write + verify dim_customer ──────────────────────────────────────────────
dim_customer_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_customer")

written = spark.table(f"{GOLD}.dim_customer")
print(f"{GOLD}.dim_customer rows : {written.count():,}")
print(f"Distinct customer_sk    : {written.select('customer_sk').distinct().count():,}")

### C3 — `dim_product` (SCD2 — already versioned by Silver)

**Source:** `gbmart.silver.products`
**Natural key:** `product_id`
**Surrogate key:** `product_sk` — **reused from Silver**, same reasoning as `dim_customer` above.

`category` / `sub_category` are what let Genie answer "revenue by category" later — and `is_current` matters just as much here, since Day 7's Gold views filter on it directly when they join `dim_product`. This dimension is a **conformed dimension** in the making — a future `fact_returns` would join to these exact same `product_sk` values, so "revenue by category" and "return rate by category" always agree on what a category is.

In [ ]:
# ─── Inspect Silver source FIRST ──────────────────────────────────────────────
silver_products = spark.table(f"{SILVER}.products")
silver_products.printSchema()
silver_products.display()

In [ ]:
# ─── Build dim_product ────────────────────────────────────────────────────────
# Same reasoning as dim_customer: product_sk / is_current / effective_start_date /
# effective_end_date already exist on silver.products (Day 5's SCD2 initial load).
# Select them as-is -- is_current especially matters, since Day 7's Gold views
# join dim_product with an explicit "is_current = true" filter.
dim_product_df = (
    silver_products
    .select(
        "product_sk", "product_id", "product_name", "category", "sub_category",
        "discounted_price_inr", "is_current", "effective_start_date", "effective_end_date"
    )
)

dim_product_df.show(5, truncate=False)

In [ ]:
# ─── Write + verify dim_product ───────────────────────────────────────────────
dim_product_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_product")

written = spark.table(f"{GOLD}.dim_product")
print(f"{GOLD}.dim_product rows : {written.count():,}")
print(f"Distinct product_sk   : {written.select('product_sk').distinct().count():,}")

### C4 — `dim_address`

**Source:** `gbmart.silver.address` (singular — not `addresses`)
**Natural key:** `address_id`
**Surrogate key:** generated fresh here in Gold — Silver never assigned this table one.

> ⚠️ **The many-to-many problem — and how it's actually resolved.** One customer can have more than one address (Billing, Shipping, or both) — so a plain `dim_address` row can't say by itself "which address did *this specific order line* ship to." That's a real design problem, but **it is not solved with a bridge table.** Day 7's `fact_sales` build resolves it with a window-function ranking: rank each customer's addresses (prefer `Shipping`), keep the top-ranked one, and join on `customer_id`. `dim_address` itself stays a plain, honest dimension — the resolution logic lives in the fact build, not here. (Day 7 HOL 1 *does* build one illustrative bridge table — for a hypothetical products↔campaigns relationship — to teach the bridge-table pattern generally. That example is unrelated to addresses.)

In [ ]:
# ─── Inspect Silver source FIRST ──────────────────────────────────────────────
# Table name is singular "address" -- matches the real gbmart workspace convention.
silver_address = spark.table(f"{SILVER}.address")
silver_address.printSchema()
silver_address.display()

In [ ]:
# ─── Build dim_address ────────────────────────────────────────────────────────
dim_address_df = (
    silver_address
    .select("address_id", "customer_id", "city", "state", "pincode", "address_type")
    .withColumn("address_sk", sha2(col("address_id").cast("string"), 256))
    .select("address_sk", "address_id", "customer_id", "city", "state", "pincode", "address_type")
)

dim_address_df.show(5, truncate=False)

In [ ]:
# ─── Write + verify dim_address ───────────────────────────────────────────────
dim_address_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_address")

written = spark.table(f"{GOLD}.dim_address")
print(f"{GOLD}.dim_address rows : {written.count():,}")
print(f"Distinct address_sk   : {written.select('address_sk').distinct().count():,}")

### C5 — `dim_orders`

**Source:** `gbmart.silver.orders`
**Natural key:** `order_id`
**Surrogate key:** generated fresh here in Gold — Silver never assigned this table one.

Order header attributes (`order_channel`, `shipping_tier_id`, `supplier_id`, `order_date`) describe the order as a whole, separately from any one line item in it — that's what makes them dimension-shaped rather than fact-shaped. Two attributes are worth calling out explicitly:

- `order_date` is also reachable through `dim_date` (via `fact_sales`), but it's kept here too since it's a natural attribute of the order itself — letting you query "this customer's orders by date" without touching the fact table at all.
- `shipping_tier_id` / `supplier_id` are carried as plain IDs, undecoded — there's no `dim_supplier` or shipping-tier-name lookup table in this course, so there's nothing to join them against yet.

> **Heads-up for Day 7:** `fact_sales` carries `Order_ID` directly as a natural key — it does **not** join through `dim_orders.order_sk`. `dim_orders` still exists as its own Gold table (built here), it's just not the fact table's chosen join path for order attributes. You'll see this exact point revisited in Day 6 ILT 3's "degenerate dimension" discussion.

In [ ]:
# ─── Inspect Silver source FIRST ──────────────────────────────────────────────
silver_orders = spark.table(f"{SILVER}.orders")
silver_orders.printSchema()
silver_orders.display()

In [ ]:
# ─── Build dim_orders ─────────────────────────────────────────────────────────
dim_orders_df = (
    silver_orders
    .select("order_id", "customer_id", "order_date", "shipping_tier_id", "supplier_id", "order_channel")
    .withColumn("order_sk", sha2(col("order_id").cast("string"), 256))
    .select("order_sk", "order_id", "customer_id", "order_date", "shipping_tier_id", "supplier_id", "order_channel")
)

dim_orders_df.show(5, truncate=False)

In [ ]:
# ─── Write + verify dim_orders ─────────────────────────────────────────────────
dim_orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_orders")

written = spark.table(f"{GOLD}.dim_orders")
print(f"{GOLD}.dim_orders rows : {written.count():,}")
print(f"Distinct order_sk    : {written.select('order_sk').distinct().count():,}")

---
## <span style="color:#5b21b6">Phase D</span> — Verify All 6 Dimensions

**Goal:** Confirm every dimension landed in Gold, has the row count you expect, and has a clean (unique, non-null) surrogate key.

In [ ]:
# ─── Verify: all 6 Gold dimensions at a glance ────────────────────────────────
dimensions = {
    "dim_customer":       "customer_sk",
    "dim_product":        "product_sk",
    "dim_date":           "date_key",
    "dim_address":        "address_sk",
    "dim_payment_method": "payment_method_sk",
    "dim_orders":         "order_sk",
}

print(f"{'TABLE':<22} {'ROWS':>10}  {'DISTINCT SK':>12}  {'SK COLUMN':<20}")
print("-" * 72)

for table, sk_col in dimensions.items():
    df       = spark.table(f"{GOLD}.{table}")
    rows     = df.count()
    distinct = df.select(sk_col).distinct().count()
    flag     = "OK" if rows == distinct else "MISMATCH — investigate duplicates"
    print(f"{table:<22} {rows:>10,}  {distinct:>12,}  {sk_col:<20}  {flag}")

**`fact_sales` is NOT built in this session.** All 6 dimensions are now ready in Gold:

- `gbmart.gold.dim_customer`
- `gbmart.gold.dim_product`
- `gbmart.gold.dim_date`
- `gbmart.gold.dim_address`
- `gbmart.gold.dim_payment_method`
- `gbmart.gold.dim_orders`

**Day 7's "Build Gold Layer" hands-on picks up from exactly here** — it builds `fact_sales` itself (one row per order line item, resolving the customer↔address one-to-many by ranking rather than a bridge table), and separately builds one illustrative bridge table (products↔campaigns) purely to teach the bridge-table pattern in general.

---
## Submission Checklist

Before uploading this notebook, fill in each blank and verify the checklist.

```
Submission Checklist
────────────────────────────────────────────────────────
✅ gbmart.gold schema created
✅ Grain sentence written (Phase A1)
✅ 6-dimension design table filled in (Phase A2)
✅ Star sketch reviewed (Phase A3)
── dim_date rows generated:              ______
── dim_payment_method rows:               ______
── dim_customer rows:                     ______
── dim_product rows:                      ______
── dim_address rows:                      ______
── dim_orders rows:                       ______
── All 6 surrogate keys unique (Phase D)? ______
✅ Confirmed: fact_sales NOT built today — that's Day 7
────────────────────────────────────────────────────────
```